Sample-size calculations (RQ1–RQ4)

Common assumptions: two-sided α = 0.05, power = 0.80, 95% confidence, intra-class correlation ρ = 0.05

The final sample size is **N = max(RQ1–RQ4)**.


In [1]:
# --- Setup: works on Google Colab and locally -------------------------------
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("VS_DATA_ROOT", "/content/drive/MyDrive/audio-detection-human-vs-ai")
    if not os.path.exists("/content/audio-detection-human-vs-ai"):
        !git clone -q https://github.com/abhilash-u/audio-detection-human-vs-ai /content/audio-detection-human-vs-ai
    %cd /content/audio-detection-human-vs-ai
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd
from src import config as C
C.ensure_dirs()
print("Data folder:", C.DATA_ROOT)

Mounted at /content/drive
/content/audio-detection-human-vs-ai
Data folder: /content/drive/MyDrive/audio-detection-human-vs-ai


In [2]:
from src import sample_size as S
for r in (S.rq1(), S.rq2(), S.rq3(), S.rq4()):
    print(r["rq"], "-", r["method"])
    for k, v in r.items():
        if k not in ("rq", "method"):
            print(f"   {k:28s} {v:.4f}" if isinstance(v, float) else f"   {k:28s} {v}")

RQ1 - Power: two independent means
   alpha                        0.0008
   d                            0.3000
   n0_per_group                 393
   m                            90.0000
   deff                         5.4500
   n_per_group                  2142
   minimum_N                    4284
   planned                      7200
   detectable_d                 0.2312
RQ2 - Power: paired proportions (McNemar)
   psi                          0.1000
   delta                        0.0300
   n0_pairs                     870
   deff                         2.0694
   minimum_N                    1801
   planned                      2500
   epv_training_min_synthetic   1280
RQ3 - Confidence interval for proportions
   n0_per_system                151
   deff_system                  1.0769
   n_per_system                 163
   n0_fpr                       457
   deff_human                   2.0694
   n_human                      946
   minimum_N                    2739
   planned     

In [3]:
table, final_n = S.summary()
display(table)
print(f"Final sample size N = max(RQ1-RQ4) = {final_n:,} clips")

,rq,method,minimum_N,planned,meets_requirement
0,RQ1,Power: two independent means,4284,7200,True
1,RQ2,Power: paired proportions (McNemar),1801,2500,True
2,RQ3,Confidence interval for proportions,2739,3370,True
3,RQ4,"Power: paired proportions (McNemar), clean vs ...",1801,4370,True


Final sample size N = max(RQ1-RQ4) = 4,284 clips


In [4]:
pd.DataFrame([{"rho": rho, **{r["rq"]: r["minimum_N"] for r in (S.rq1(rho), S.rq2(rho), S.rq3(rho), S.rq4(rho))},
               "final_N": S.summary(rho)[1]} for rho in (0.01, 0.03, 0.05, 0.08, 0.10)])

,rho,RQ1,RQ2,RQ3,RQ4,final_N
0,0.01,1486,1057,2249,1057,2249
1,0.03,2886,1429,2489,1429,2886
2,0.05,4284,1801,2739,1801,4284
3,0.08,6384,2359,3109,2359,6384
4,0.10,7782,2731,3360,2731,7782


In [5]:
pd.DataFrame([{"d": d, "alpha=.05": S.ceil(S.two_means_n(d)), "alpha=.05/64": S.ceil(S.two_means_n(d, 0.05/64))}
              for d in (0.8, 0.5, 0.3, 0.2)])

,d,alpha=.05,alpha=.05/64
0,0.8,25,56
1,0.5,63,142
2,0.3,175,393
3,0.2,393,883
